# PA3 Task 3 on Kaggle — live parallel sampling vs sequential refinement (step 4)

Runs `task3.live` for the fixed 50 MATH questions (exported in the repo at `outputs/task3/live/questions.json`):
Qwen2.5-3B-Instruct (pinned revision), FP16, batch size 1. Questions are split over the two T4s
(even / odd question index); every strategy for a question runs on the same GPU.

**Before running:** right-hand panel → *Session options* → **Accelerator: GPU T4 x2**, **Internet: On**. No inputs needed.
Then **Save Version → Save & Run All (Commit)** (about 1–2 h).

Afterwards: saved version → **Output** → download `task3_live.zip` and unzip it into `outputs/task3/live/` locally,
then run `python -m task3.live_analyze`.

In [ ]:
!git clone -q https://github.com/ahmadhassann96/ATML_PA3.git /kaggle/working/ATML_PA3
%cd /kaggle/working/ATML_PA3
!git log --oneline -1
!pip install -q "transformers==4.57.1"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
import os
os.environ["PA3_OUT"] = "/kaggle/working/outputs"
os.environ["HF_HUB_DISABLE_XET"] = "1"
# outputs/task3/live/questions.json comes from the repo; copy it next to the run outputs
!mkdir -p /kaggle/working/outputs/task3/live && cp outputs/task3/live/questions.json /kaggle/working/outputs/task3/live/
# download the pinned model once, before the two GPU workers start
from huggingface_hub import snapshot_download
from task3 import config as C
print(snapshot_download(C.LIVE_MODEL, revision=C.LIVE_REVISION))

In [ ]:
# one worker per T4; logs: /kaggle/working/live_shard<k>.log
import subprocess, sys, threading, time

def run(shard):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(shard))
    with open(f"/kaggle/working/live_shard{shard}.log", "w") as f:
        rc = subprocess.run([sys.executable, "-m", "task3.live", "--shard", str(shard), "--n_shards", "2"],
                            stdout=f, stderr=subprocess.STDOUT, env=env).returncode
    print(f"shard {shard} finished with exit code {rc}", flush=True)

threads = [threading.Thread(target=run, args=(k,)) for k in range(2)]
for t in threads:
    t.start()
while any(t.is_alive() for t in threads):
    time.sleep(300)
    for k in range(2):
        p = f"/kaggle/working/live_shard{k}.log"
        if os.path.exists(p):
            lines = open(p, errors="ignore").read().strip().splitlines()
            print(time.strftime("%H:%M"), f"shard {k}:", lines[-1][:160] if lines else "", flush=True)
for t in threads:
    t.join()

In [ ]:
!wc -l /kaggle/working/outputs/task3/live/calls_shard*.jsonl
!cd /kaggle/working/outputs/task3/live && zip -q /kaggle/working/task3_live.zip calls_shard*.jsonl run_info_shard*.json
!ls -lh /kaggle/working/task3_live.zip